# PubChem Fetch -- Example

![PubChem Fetch -- Example](https://proto-bio.github.io/proto-assets/images/tool/pubchem/hero.png)

This notebook demonstrates `run_pubchem_fetch`, which resolves one of CID, name, SMILES, InChI, or InChIKey to canonical small-molecule structure data via the [PubChem PUG REST API](https://pubchem.ncbi.nlm.nih.gov/). Given any single identifier, the tool resolves it to a PubChem CID, fetches a configurable property bundle (formula, weight, SMILES, InChI, descriptors), and optionally returns a list of synonyms. See [Kim et al. 2023](https://doi.org/10.1093/nar/gkac956) for the underlying database.

In [ ]:
from proto_tools.tools.database_retrieval import (
    PubChemFetchConfig,
    PubChemFetchInput,
    run_pubchem_fetch,
)
from proto_tools.utils.notebook_docs import display_api_reference

## Input API Reference

Exactly one of the five identifier fields below must be provided.

In [ ]:
# Display input docs
display_api_reference("pubchem", "input")

## Config API Reference

In [ ]:
# Display config docs
display_api_reference("pubchem", "config")

The default `properties` bundle covers structure, mass, and basic descriptor counts: `Title`, `MolecularFormula`, `MolecularWeight`, `SMILES`, `ConnectivitySMILES`, `InChI`, `InChIKey`, `IUPACName`, `ExactMass`, `TPSA`, `Complexity`, `Charge`, `HBondDonorCount`, `HBondAcceptorCount`, `RotatableBondCount`, `HeavyAtomCount`.

## Output API Reference

In [ ]:
# Display output docs
display_api_reference("pubchem", "output")

## Lookup by name

The most common starting point: pass a common name and let PubChem resolve it. Here we fetch aspirin and inspect the canonical structure plus a few descriptors.

In [ ]:
inputs = PubChemFetchInput(name="aspirin")
config = PubChemFetchConfig()
output = run_pubchem_fetch(inputs, config)

print(f"CID: {output.cid}")
print(f"Molecular formula: {output.molecular_formula}")
print(f"Molecular weight: {output.molecular_weight}")
print(f"SMILES: {output.smiles}")
print(f"InChIKey: {output.inchikey}")
print(f"IUPAC name: {output.iupac_name}")
print(f"TPSA: {output.tpsa}")
print(f"H-bond donors: {output.hbond_donor_count}")
print(f"H-bond acceptors: {output.hbond_acceptor_count}")

## Lookup by CID

When you already know the PubChem CID, pass it directly to skip the resolver step. Here we fetch caffeine (CID 2519).

In [ ]:
caffeine_output = run_pubchem_fetch(PubChemFetchInput(cid=2519), PubChemFetchConfig())

print(f"CID: {caffeine_output.cid}")
print(f"Molecular formula: {caffeine_output.molecular_formula}")
print(f"Molecular weight: {caffeine_output.molecular_weight}")
print(f"SMILES: {caffeine_output.smiles}")
print(f"IUPAC name: {caffeine_output.iupac_name}")

## Lookup by SMILES

PubChem can resolve a SMILES string back to its registered compound record. Here we round-trip aspirin's SMILES and verify the resolved CID.

In [ ]:
smiles_output = run_pubchem_fetch(
    PubChemFetchInput(smiles="CC(=O)Oc1ccccc1C(=O)O"),
    PubChemFetchConfig(),
)

print(f"Resolved CID: {smiles_output.cid}")
print(f"Molecular formula: {smiles_output.molecular_formula}")
assert smiles_output.cid == 2244, f"Expected CID 2244 for aspirin, got {smiles_output.cid}"

## Lookup by InChIKey

InChIKey is a fixed-length hash of the InChI string and is the most reliable identifier for exact-structure matching. Here we look up aspirin again via its InChIKey.

In [ ]:
inchikey_output = run_pubchem_fetch(
    PubChemFetchInput(inchikey="BSYNRYMUTXBXSQ-UHFFFAOYSA-N"),
    PubChemFetchConfig(),
)

print(f"Resolved CID: {inchikey_output.cid}")
print(f"IUPAC name: {inchikey_output.iupac_name}")
assert inchikey_output.cid == 2244, f"Expected CID 2244 for aspirin, got {inchikey_output.cid}"

## Fetch with synonyms

Setting `include_synonyms=True` issues an extra HTTP request for the compound's synonym list. PubChem can return hundreds of synonyms per compound; the wrapper caps the returned list at 50. This is handy when you need to map between trade names, brand names, and chemical identifiers.

In [ ]:
synonyms_output = run_pubchem_fetch(
    PubChemFetchInput(name="aspirin"),
    PubChemFetchConfig(include_synonyms=True),
)

print(f"CID: {synonyms_output.cid}")
print(f"Synonyms ({len(synonyms_output.synonyms)} of up to 50):")
for synonym in synonyms_output.synonyms[:10]:
    print(f"  - {synonym}")

## Custom property subset

When you only need a few fields, restrict the `properties` list to keep the response small. Here we ask only for formula, weight, and SMILES.

In [ ]:
subset_output = run_pubchem_fetch(
    PubChemFetchInput(name="aspirin"),
    PubChemFetchConfig(properties=["MolecularFormula", "MolecularWeight", "SMILES"]),
)

print(f"Molecular formula: {subset_output.molecular_formula}")
print(f"Molecular weight: {subset_output.molecular_weight}")
print(f"SMILES: {subset_output.smiles}")

## Export Results

Fetched results can be saved to JSON format for downstream use in other tools or analysis pipelines.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("aspirin_pubchem", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'aspirin_pubchem.json'}")